In [13]:
# ============================================================
# CELL 1 — PROJECT CONFIGURATION & DIRECTORY SETUP
# ============================================================

from pathlib import Path
import json
import os
import gc

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# ------------------------------------------------------------
# 2. Directory structure
# ------------------------------------------------------------

DIRS = {
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "data_chunks": PROJECT_ROOT / "data" / "chunks",
    "embeddings": PROJECT_ROOT / "embeddings",
    "vectorstore": PROJECT_ROOT / "vectorstore",
    "evaluation": PROJECT_ROOT / "evaluation",
    "results": PROJECT_ROOT / "results",
    "logs": PROJECT_ROOT / "logs",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

# Create directories safely
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Dataset configuration
# ------------------------------------------------------------

DATASET_FILENAME = "synthetic_knowledge_items.csv"

DATASET_PATH = Path("E:/rag/synthetic_knowledge_items.csv")

# ------------------------------------------------------------
# 4. RAG baseline configuration
# ------------------------------------------------------------

CONFIG = {
    "project": {
        "name": "Synthetic IT Knowledge RAG",
        "dataset_name": "Synthetic IT-Related Knowledge Items",
        "dataset_version": "Version 3",
        "expected_records": 100,
        "expected_columns": 4,
    },

    "dataset": {
        "filename": DATASET_FILENAME,
        "raw_path": str(DATASET_PATH),
    },

    "chunking": {
        "chunk_size": 500,
        "chunk_overlap": 50,
    },

    "retrieval": {
        "method": "dense",
        "top_k": 5,
    },

    "generation": {
        "provider": "ollama",
        "base_url": "http://localhost:11434",
        "model": None,  # Set after checking installed Ollama models
    },
}

# ------------------------------------------------------------
# 5. Persistent configuration checkpoint
# ------------------------------------------------------------

CONFIG_CHECKPOINT = DIRS["checkpoints"] / "project_config.json"

try:
    with open(CONFIG_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=4)

    print("[SUCCESS] Project configuration saved.")
    
except Exception as e:
    print("[ERROR] Failed to save project configuration.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise

# ------------------------------------------------------------
# 6. Environment summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG PROJECT INITIALIZATION")
print("=" * 60)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Dataset path       : {DATASET_PATH}")
print(f"Dataset expected   : {CONFIG['project']['expected_records']} records")
print(f"Expected columns   : {CONFIG['project']['expected_columns']}")
print(f"Chunk size         : {CONFIG['chunking']['chunk_size']}")
print(f"Chunk overlap      : {CONFIG['chunking']['chunk_overlap']}")
print(f"Retrieval method   : {CONFIG['retrieval']['method']}")
print(f"Top-K              : {CONFIG['retrieval']['top_k']}")
print(f"Ollama URL         : {CONFIG['generation']['base_url']}")

print("=" * 60)
print("[SUCCESS] Initialization completed.")

[SUCCESS] Project configuration saved.

RAG PROJECT INITIALIZATION
Project root       : e:\rag
Dataset path       : E:\rag\synthetic_knowledge_items.csv
Dataset expected   : 100 records
Expected columns   : 4
Chunk size         : 500
Chunk overlap      : 50
Retrieval method   : dense
Top-K              : 5
Ollama URL         : http://localhost:11434
[SUCCESS] Initialization completed.


In [14]:
# ============================================================
# CELL 2 — OLLAMA CONNECTION & MODEL DISCOVERY
# ============================================================

import requests
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Ollama configuration
# ------------------------------------------------------------

OLLAMA_BASE_URL = CONFIG["generation"]["base_url"]
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

OLLAMA_CHECKPOINT = (
    DIRS["checkpoints"] / "ollama_environment.json"
)

# ------------------------------------------------------------
# 2. Check Ollama server
# ------------------------------------------------------------

print("=" * 60)
print("OLLAMA CONNECTION CHECK")
print("=" * 60)

try:
    response = requests.get(
        OLLAMA_TAGS_URL,
        timeout=10
    )

    response.raise_for_status()

    ollama_data = response.json()

    print("[SUCCESS] Ollama server is running.")
    print(f"[INFO] Ollama URL: {OLLAMA_BASE_URL}")

except requests.exceptions.ConnectionError:
    print("[ERROR] Could not connect to Ollama.")
    print(f"[ERROR] Expected Ollama at: {OLLAMA_BASE_URL}")
    print()
    print("Make sure Ollama is running on your computer.")
    raise

except requests.exceptions.Timeout:
    print("[ERROR] Ollama connection timed out.")
    print(f"[ERROR] URL: {OLLAMA_BASE_URL}")
    raise

except requests.exceptions.HTTPError as e:
    print("[ERROR] Ollama returned an HTTP error.")
    print(f"[ERROR] Status code: {response.status_code}")
    print(f"[ERROR] Details: {e}")
    raise

except Exception as e:
    print("[ERROR] Unexpected Ollama connection error.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 3. Extract installed models
# ------------------------------------------------------------

try:
    installed_models = ollama_data.get("models", [])

    if not installed_models:
        raise RuntimeError(
            "Ollama is running, but no local models were found."
        )

    model_names = [
        model.get("name")
        for model in installed_models
        if model.get("name")
    ]

    print(f"\n[INFO] Installed Ollama models: {len(model_names)}")

    for index, model_name in enumerate(model_names, start=1):
        print(f"  {index}. {model_name}")

except Exception as e:
    print("[ERROR] Failed to inspect Ollama models.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

# ------------------------------------------------------------
# 4. Display model information
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MODEL INFORMATION")
print("-" * 60)

for model in installed_models:
    name = model.get("name", "Unknown")
    size = model.get("size", 0)

    # Convert bytes → GB
    size_gb = size / (1024 ** 3) if size else 0

    print(f"Model : {name}")
    print(f"Size  : {size_gb:.2f} GB")
    print()

# ------------------------------------------------------------
# 5. Save Ollama environment checkpoint
# ------------------------------------------------------------

ollama_environment = {
    "base_url": OLLAMA_BASE_URL,
    "server_status": "connected",
    "installed_models": model_names,
    "model_details": installed_models,
}

try:
    with open(
        OLLAMA_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            ollama_environment,
            f,
            indent=4
        )

    print("[SUCCESS] Ollama environment checkpoint saved.")
    print(f"[INFO] Checkpoint: {OLLAMA_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save Ollama checkpoint.")
    print(f"[ERROR] Error type: {type(e).__name__}")
    print(f"[ERROR] Details: {e}")
    raise

print("=" * 60)
print("[SUCCESS] Ollama environment is ready.")
print("=" * 60)

OLLAMA CONNECTION CHECK
[SUCCESS] Ollama server is running.
[INFO] Ollama URL: http://localhost:11434

[INFO] Installed Ollama models: 5
  1. qwen3:8b
  2. qwen3.8:27b-mtp-q4_K_M
  3. qwen3.8:27b
  4. gemma4:12b
  5. gemma4:26b

------------------------------------------------------------
MODEL INFORMATION
------------------------------------------------------------
Model : qwen3:8b
Size  : 4.87 GB

Model : qwen3.8:27b-mtp-q4_K_M
Size  : 16.52 GB

Model : qwen3.8:27b
Size  : 16.52 GB

Model : gemma4:12b
Size  : 7.04 GB

Model : gemma4:26b
Size  : 16.75 GB

[SUCCESS] Ollama environment checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\ollama_environment.json
[SUCCESS] Ollama environment is ready.


In [15]:
# ============================================================
# CELL 3 — DATASET LOADING & INITIAL VALIDATION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify dataset path
# ------------------------------------------------------------

print("=" * 60)
print("DATASET LOADING & VALIDATION")
print("=" * 60)

try:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found at: {DATASET_PATH}"
        )

    if not DATASET_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset path is not a file: {DATASET_PATH}"
        )

    print(f"[SUCCESS] Dataset found:")
    print(f"         {DATASET_PATH}")

except Exception as e:
    print("[ERROR] Dataset verification failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

try:
    df_raw = pd.read_csv(DATASET_PATH)

    if df_raw.empty:
        raise ValueError("Dataset is empty.")

    print("\n[SUCCESS] Dataset loaded successfully.")

except Exception as e:
    print("[ERROR] Failed to load dataset.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Basic dimensions
# ------------------------------------------------------------

try:
    n_rows, n_columns = df_raw.shape

    print("\n" + "-" * 60)
    print("DATASET DIMENSIONS")
    print("-" * 60)

    print(f"Rows    : {n_rows:,}")
    print(f"Columns : {n_columns}")

except Exception as e:
    print("[ERROR] Failed to inspect dataset dimensions.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Validate expected dimensions
# ------------------------------------------------------------

EXPECTED_ROWS = CONFIG["project"]["expected_records"]
EXPECTED_COLUMNS = CONFIG["project"]["expected_columns"]

if n_rows != EXPECTED_ROWS:
    print(
        f"[WARNING] Expected {EXPECTED_ROWS} rows "
        f"but found {n_rows}."
    )
else:
    print(f"[SUCCESS] Record count matches expected value: {EXPECTED_ROWS}")

if n_columns != EXPECTED_COLUMNS:
    print(
        f"[WARNING] Expected {EXPECTED_COLUMNS} columns "
        f"but found {n_columns}."
    )
else:
    print(
        f"[SUCCESS] Column count matches expected value: "
        f"{EXPECTED_COLUMNS}"
    )


# ------------------------------------------------------------
# 5. Exact column names
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("COLUMN INFORMATION")
print("-" * 60)

for index, column in enumerate(df_raw.columns, start=1):
    print(f"{index}. {column}")


# ------------------------------------------------------------
# 6. Data types
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DATA TYPES")
print("-" * 60)

print(df_raw.dtypes)


# ------------------------------------------------------------
# 7. Missing-value analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MISSING VALUE ANALYSIS")
print("-" * 60)

missing_counts = df_raw.isna().sum()

missing_table = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.values,
    "missing_percentage": (
        missing_counts.values / len(df_raw) * 100
    )
})

print(missing_table.to_string(index=False))


# ------------------------------------------------------------
# 8. Duplicate analysis
# ------------------------------------------------------------

try:
    duplicate_count = df_raw.duplicated().sum()

    print("\n" + "-" * 60)
    print("DUPLICATE ANALYSIS")
    print("-" * 60)

    print(f"Duplicate rows: {duplicate_count:,}")

except Exception as e:
    print("[ERROR] Duplicate analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Sample records
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("SAMPLE RECORDS")
print("-" * 60)

display(df_raw.head())


# ------------------------------------------------------------
# 10. Dataset memory usage
# ------------------------------------------------------------

memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)

print("\n" + "-" * 60)
print("MEMORY USAGE")
print("-" * 60)

print(f"DataFrame memory usage: {memory_mb:.2f} MB")


# ------------------------------------------------------------
# 11. Create dataset inspection checkpoint
# ------------------------------------------------------------

dataset_inspection = {
    "dataset_path": str(DATASET_PATH),
    "dataset_name": CONFIG["project"]["dataset_name"],
    "dataset_version": CONFIG["project"]["dataset_version"],
    "rows": int(n_rows),
    "columns": int(n_columns),
    "column_names": list(df_raw.columns),
    "data_types": {
        column: str(dtype)
        for column, dtype in df_raw.dtypes.items()
    },
    "missing_values": {
        column: int(count)
        for column, count in missing_counts.items()
    },
    "duplicate_rows": int(duplicate_count),
    "memory_mb": round(memory_mb, 4),
}

DATASET_INSPECTION_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_inspection.json"
)

try:
    with open(
        DATASET_INSPECTION_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            dataset_inspection,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset inspection checkpoint saved.")
    print(
        f"[INFO] Checkpoint: "
        f"{DATASET_INSPECTION_CHECKPOINT}"
    )

except Exception as e:
    print("[ERROR] Failed to save dataset inspection checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 12. Final validation summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET VALIDATION SUMMARY")
print("=" * 60)

print(f"Records              : {n_rows:,}")
print(f"Columns              : {n_columns}")
print(f"Missing values       : {int(missing_counts.sum()):,}")
print(f"Duplicate rows       : {duplicate_count:,}")
print(f"Memory usage         : {memory_mb:.2f} MB")
print(f"Original CSV modified: NO")

print("=" * 60)
print("[SUCCESS] Dataset inspection completed.")
print("=" * 60)

DATASET LOADING & VALIDATION
[SUCCESS] Dataset found:
         E:\rag\synthetic_knowledge_items.csv

[SUCCESS] Dataset loaded successfully.

------------------------------------------------------------
DATASET DIMENSIONS
------------------------------------------------------------
Rows    : 100
Columns : 4
[SUCCESS] Record count matches expected value: 100
[SUCCESS] Column count matches expected value: 4

------------------------------------------------------------
COLUMN INFORMATION
------------------------------------------------------------
1. ki_topic
2. ki_text
3. alt_ki_text
4. bad_ki_text

------------------------------------------------------------
DATA TYPES
------------------------------------------------------------
ki_topic       str
ki_text        str
alt_ki_text    str
bad_ki_text    str
dtype: object

------------------------------------------------------------
MISSING VALUE ANALYSIS
------------------------------------------------------------
     column  missing_count 

,ki_topic,ki_text,alt_ki_text,bad_ki_text
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"To set up a mobile device for company email, f...",# Setting Up a Mobile Device for Company Email...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"If you have forgotten your PIN, you can reset ...","# How to Resetting Your Forgot PIN \n\nSo, you..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,To configure VPN access for remote workers at ...,# How to Set Up VPN Access for Remote Workrs\n...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,When troubleshooting issues with Microsoft Off...,# Troubleshooting Issues with Microsoft Office...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...","To set up a conference call on Cisco Webex, fo...",# How To Set Up A Conference Call on Cisco Web...



------------------------------------------------------------
MEMORY USAGE
------------------------------------------------------------
DataFrame memory usage: 1.35 MB

[SUCCESS] Dataset inspection checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_inspection.json

DATASET VALIDATION SUMMARY
Records              : 100
Columns              : 4
Missing values       : 0
Duplicate rows       : 0
Memory usage         : 1.35 MB
Original CSV modified: NO
[SUCCESS] Dataset inspection completed.


In [16]:
# ============================================================
# CELL 4 — DATASET PROFILING & RAG FIELD DEFINITION
# ============================================================

import pandas as pd
import json
from pathlib import Path

print("=" * 60)
print("DATASET PROFILING & RAG FIELD DEFINITION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Define the dataset schema
# ------------------------------------------------------------

RAG_SCHEMA = {
    "document_id_source": "row_index",

    "topic_column": "ki_topic",

    "primary_text_column": "ki_text",

    "alternative_text_column": "alt_ki_text",

    "bad_text_column": "bad_ki_text",
}


# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_columns = [
    RAG_SCHEMA["topic_column"],
    RAG_SCHEMA["primary_text_column"],
    RAG_SCHEMA["alternative_text_column"],
    RAG_SCHEMA["bad_text_column"],
]

try:
    missing_columns = [
        column
        for column in required_columns
        if column not in df_raw.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}"
        )

    print("[SUCCESS] All required dataset columns are present.")

except Exception as e:
    print("[ERROR] Dataset schema validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 3. Create a non-destructive profiling copy
# ------------------------------------------------------------

try:
    df_profile = df_raw.copy(deep=True)

    print("[SUCCESS] Profiling copy created.")
    print("[INFO] Original dataset remains unchanged.")

except Exception as e:
    print("[ERROR] Failed to create profiling copy.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Calculate character lengths
# ------------------------------------------------------------

try:
    df_profile["topic_char_length"] = (
        df_profile[RAG_SCHEMA["topic_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["primary_char_length"] = (
        df_profile[RAG_SCHEMA["primary_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["alternative_char_length"] = (
        df_profile[RAG_SCHEMA["alternative_text_column"]]
        .astype(str)
        .str.len()
    )

    df_profile["bad_char_length"] = (
        df_profile[RAG_SCHEMA["bad_text_column"]]
        .astype(str)
        .str.len()
    )

    print("[SUCCESS] Text-length profiling completed.")

except Exception as e:
    print("[ERROR] Text-length calculation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Generate descriptive statistics
# ------------------------------------------------------------

length_columns = [
    "topic_char_length",
    "primary_char_length",
    "alternative_char_length",
    "bad_char_length",
]

try:
    length_statistics = (
        df_profile[length_columns]
        .describe()
        .round(2)
    )

    print("\n" + "-" * 60)
    print("TEXT LENGTH STATISTICS")
    print("-" * 60)

    display(length_statistics)

except Exception as e:
    print("[ERROR] Failed to calculate text statistics.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Display min/max examples
# ------------------------------------------------------------

try:
    print("\n" + "-" * 60)
    print("TEXT LENGTH RANGE")
    print("-" * 60)

    for column in length_columns:
        print(
            f"{column:25s}: "
            f"min={df_profile[column].min():,} | "
            f"max={df_profile[column].max():,} | "
            f"mean={df_profile[column].mean():,.2f}"
        )

except Exception as e:
    print("[ERROR] Failed to display length ranges.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Check whether text fields are actually distinct
# ------------------------------------------------------------

try:
    primary_equals_alternative = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["alternative_text_column"]]
    ).sum()

    primary_equals_bad = (
        df_raw[RAG_SCHEMA["primary_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    alternative_equals_bad = (
        df_raw[RAG_SCHEMA["alternative_text_column"]]
        ==
        df_raw[RAG_SCHEMA["bad_text_column"]]
    ).sum()

    print("\n" + "-" * 60)
    print("TEXT VERSION COMPARISON")
    print("-" * 60)

    print(
        f"Primary == Alternative : "
        f"{primary_equals_alternative:,} / {len(df_raw):,}"
    )

    print(
        f"Primary == Bad         : "
        f"{primary_equals_bad:,} / {len(df_raw):,}"
    )

    print(
        f"Alternative == Bad     : "
        f"{alternative_equals_bad:,} / {len(df_raw):,}"
    )

except Exception as e:
    print("[ERROR] Text version comparison failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 8. Inspect topics
# ------------------------------------------------------------

try:
    unique_topics = df_raw[
        RAG_SCHEMA["topic_column"]
    ].nunique()

    print("\n" + "-" * 60)
    print("TOPIC ANALYSIS")
    print("-" * 60)

    print(f"Total records : {len(df_raw):,}")
    print(f"Unique topics : {unique_topics:,}")

    if unique_topics != len(df_raw):
        print(
            "[WARNING] Multiple records share the same topic."
        )
    else:
        print(
            "[SUCCESS] Every knowledge item has a unique topic."
        )

except Exception as e:
    print("[ERROR] Topic analysis failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 9. Define baseline RAG document strategy
# ------------------------------------------------------------

DOCUMENT_STRATEGY = {
    "baseline_corpus": "primary",
    "topic_field": RAG_SCHEMA["topic_column"],
    "content_field": RAG_SCHEMA["primary_text_column"],

    "excluded_from_baseline": [
        RAG_SCHEMA["alternative_text_column"],
        RAG_SCHEMA["bad_text_column"],
    ],

    "alternative_text_usage": (
        "Reserved for controlled experiments/evaluation."
    ),

    "bad_text_usage": (
        "Reserved for robustness/adversarial experiments "
        "and evaluation."
    ),
}


# ------------------------------------------------------------
# 10. Save schema and profiling checkpoint
# ------------------------------------------------------------

PROFILE_CHECKPOINT = (
    DIRS["checkpoints"] / "dataset_profile.json"
)

profile_checkpoint = {
    "schema": RAG_SCHEMA,
    "document_strategy": DOCUMENT_STRATEGY,

    "record_count": int(len(df_raw)),
    "unique_topics": int(unique_topics),

    "text_statistics": {
        column: {
            metric: float(length_statistics.loc[metric, column])
            for metric in length_statistics.index
        }
        for column in length_statistics.columns
    },

    "version_comparison": {
        "primary_equals_alternative": int(
            primary_equals_alternative
        ),
        "primary_equals_bad": int(
            primary_equals_bad
        ),
        "alternative_equals_bad": int(
            alternative_equals_bad
        ),
    },
}

try:
    with open(
        PROFILE_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            profile_checkpoint,
            f,
            indent=4
        )

    print("\n[SUCCESS] Dataset profile checkpoint saved.")
    print(f"[INFO] Checkpoint: {PROFILE_CHECKPOINT}")

except Exception as e:
    print("[ERROR] Failed to save dataset profile checkpoint.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 11. Final strategy summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BASELINE RAG DOCUMENT STRATEGY")
print("=" * 60)

print(f"Topic field       : {RAG_SCHEMA['topic_column']}")
print(f"Primary text      : {RAG_SCHEMA['primary_text_column']}")
print(f"Alternative text  : {RAG_SCHEMA['alternative_text_column']}")
print(f"Bad text          : {RAG_SCHEMA['bad_text_column']}")

print("\nBaseline corpus:")
print("  ki_topic + ki_text")

print("\nExcluded from baseline:")
print("  alt_ki_text")
print("  bad_ki_text")

print("\n[SUCCESS] Dataset profiling completed.")
print("=" * 60)

DATASET PROFILING & RAG FIELD DEFINITION
[SUCCESS] All required dataset columns are present.
[SUCCESS] Profiling copy created.
[INFO] Original dataset remains unchanged.
[SUCCESS] Text-length profiling completed.

------------------------------------------------------------
TEXT LENGTH STATISTICS
------------------------------------------------------------


,topic_char_length,primary_char_length,alternative_char_length,bad_char_length
count,100.00,100.00,100.00,100.00
mean,41.55,2581.34,2440.64,2921.60
std,9.77,361.60,372.04,319.16
min,25.00,1606.00,1526.00,2326.00
25%,33.00,2327.25,2240.50,2756.25
50%,41.00,2573.50,2415.00,2923.00
75%,50.00,2824.25,2660.50,3108.00
max,64.00,3730.00,3456.00,3904.00



------------------------------------------------------------
TEXT LENGTH RANGE
------------------------------------------------------------
topic_char_length        : min=25 | max=64 | mean=41.55
primary_char_length      : min=1,606 | max=3,730 | mean=2,581.34
alternative_char_length  : min=1,526 | max=3,456 | mean=2,440.64
bad_char_length          : min=2,326 | max=3,904 | mean=2,921.60

------------------------------------------------------------
TEXT VERSION COMPARISON
------------------------------------------------------------
Primary == Alternative : 0 / 100
Primary == Bad         : 0 / 100
Alternative == Bad     : 0 / 100

------------------------------------------------------------
TOPIC ANALYSIS
------------------------------------------------------------
Total records : 100
Unique topics : 98
[WARNING] Multiple records share the same topic.

[SUCCESS] Dataset profile checkpoint saved.
[INFO] Checkpoint: e:\rag\checkpoints\dataset_profile.json

BASELINE RAG DOCUMENT STRATEGY


In [17]:
# ============================================================
# CELL 5 — RAG DOCUMENT CREATION & PERSISTENCE
# ============================================================

import json
import re
import gc
from pathlib import Path

print("=" * 60)
print("RAG DOCUMENT CREATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TOPIC_COLUMN = RAG_SCHEMA["topic_column"]
TEXT_COLUMN = RAG_SCHEMA["primary_text_column"]

DOCUMENTS_CHECKPOINT = (
    DIRS["data_processed"] / "rag_documents.json"
)


# ------------------------------------------------------------
# 2. Text cleaning function
# ------------------------------------------------------------

def clean_document_text(text):
    """
    Perform conservative text cleaning.

    The purpose is to remove accidental formatting noise
    while preserving the actual knowledge content.
    """

    if text is None:
        return ""

    text = str(text)

    # Normalize Windows line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing whitespace from each line
    text = "\n".join(
        line.rstrip()
        for line in text.split("\n")
    )

    # Collapse excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    # Remove unnecessary leading/trailing whitespace
    text = text.strip()

    return text


# ------------------------------------------------------------
# 3. Validate source data before document creation
# ------------------------------------------------------------

try:
    if df_raw.empty:
        raise ValueError(
            "Source DataFrame is empty."
        )

    if TOPIC_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Topic column '{TOPIC_COLUMN}' not found."
        )

    if TEXT_COLUMN not in df_raw.columns:
        raise KeyError(
            f"Text column '{TEXT_COLUMN}' not found."
        )

    if df_raw[TOPIC_COLUMN].isna().any():
        raise ValueError(
            "Topic column contains missing values."
        )

    if df_raw[TEXT_COLUMN].isna().any():
        raise ValueError(
            "Primary text column contains missing values."
        )

    print("[SUCCESS] Source dataset validation passed.")

except Exception as e:
    print("[ERROR] Source validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 4. Create RAG documents
# ------------------------------------------------------------

rag_documents = []

try:

    for row_index, row in df_raw.iterrows():

        topic = str(row[TOPIC_COLUMN]).strip()
        original_text = str(row[TEXT_COLUMN])

        cleaned_text = clean_document_text(
            original_text
        )

        # Validate cleaned content
        if not topic:
            raise ValueError(
                f"Empty topic at row {row_index}."
            )

        if not cleaned_text:
            raise ValueError(
                f"Empty document text at row {row_index}."
            )

        document_id = f"doc_{row_index + 1:03d}"

        document = {
            "document_id": document_id,

            "metadata": {
                "topic": topic,
                "source": CONFIG["project"]["dataset_name"],
                "dataset_version": (
                    CONFIG["project"]["dataset_version"]
                ),
                "original_row_index": int(row_index),
            },

            "text": cleaned_text,
        }

        rag_documents.append(document)

    print(
        f"[SUCCESS] Created {len(rag_documents):,} "
        f"RAG documents."
    )

except Exception as e:
    print("[ERROR] RAG document creation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 5. Validate generated documents
# ------------------------------------------------------------

try:

    if len(rag_documents) != len(df_raw):
        raise ValueError(
            "Document count does not match source "
            "record count."
        )

    document_ids = [
        document["document_id"]
        for document in rag_documents
    ]

    if len(document_ids) != len(set(document_ids)):
        raise ValueError(
            "Duplicate document IDs detected."
        )

    for document in rag_documents:

        required_keys = {
            "document_id",
            "metadata",
            "text",
        }

        if not required_keys.issubset(
            document.keys()
        ):
            raise ValueError(
                f"Invalid document structure: "
                f"{document.get('document_id')}"
            )

        if not document["text"].strip():
            raise ValueError(
                f"Empty text in "
                f"{document['document_id']}"
            )

        if not document["metadata"]["topic"].strip():
            raise ValueError(
                f"Empty topic in "
                f"{document['document_id']}"
            )

    print("[SUCCESS] Document validation passed.")
    print(f"[INFO] Valid documents: {len(rag_documents):,}")

except Exception as e:
    print("[ERROR] Document validation failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 6. Calculate document statistics
# ------------------------------------------------------------

try:

    document_lengths = [
        len(document["text"])
        for document in rag_documents
    ]

    total_characters = sum(
        document_lengths
    )

    average_length = (
        total_characters / len(document_lengths)
    )

    min_length = min(document_lengths)
    max_length = max(document_lengths)

    print("\n" + "-" * 60)
    print("PREPARED DOCUMENT STATISTICS")
    print("-" * 60)

    print(
        f"Documents           : "
        f"{len(rag_documents):,}"
    )

    print(
        f"Total characters    : "
        f"{total_characters:,}"
    )

    print(
        f"Average characters  : "
        f"{average_length:,.2f}"
    )

    print(
        f"Minimum characters  : "
        f"{min_length:,}"
    )

    print(
        f"Maximum characters  : "
        f"{max_length:,}"
    )

except Exception as e:
    print("[ERROR] Document statistics failed.")
    print(f"Error type: {type(e).__name__}")
    print(f"Error details: {e}")
    raise


# ------------------------------------------------------------
# 7. Preview prepared documents
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DOCUMENT PREVIEW")
print("-" * 60)

for document in rag_documents[:3]:

    print(
        f"\nDocument ID : "
        f"{document['document_id']}"
    )

    print(
        f"Topic       : "
        f"{document['metadata']['topic']}"
    )

    preview = document["text"][:500]

    print("Text preview:")
    print(preview)

    if len(document["text"]) > 500:
        print("...")


# ------------------------------------------------------------
# 8. Save documents using temporary file
# ------------------------------------------------------------

TEMP_DOCUMENTS_CHECKPOINT = (
    DOCUMENTS_CHECKPOINT.with_suffix(".tmp")
)

try:

    with open(
        TEMP_DOCUMENTS_CHECKPOINT,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            rag_documents,
            f,
            ensure_ascii=False,
            indent=2
        )

    # Replace old checkpoint only after successful write
    TEMP_DOCUMENTS_CHECKPOINT.replace(
        DOCUMENTS_CHECKPOINT
    )

    print(
        "\n[SUCCESS] RAG documents checkpoint saved."
    )

    print(
        f"[INFO] Path: {DOCUMENTS_CHECKPOINT}"
    )

except Exception as e:

    # Remove incomplete temporary file
    if TEMP_DOCUMENTS_CHECKPOINT.exists():
        TEMP_DOCUMENTS_CHECKPOINT.unlink(
            missing_ok=True
        )

    print(
        "[ERROR] Failed to save RAG document checkpoint."
    )

    print(
        f"Error type: {type(e).__name__}"
    )

    print(
        f"Error details: {e}"
    )

    raise


# ------------------------------------------------------------
# 9. Final integrity summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RAG DOCUMENT PREPARATION COMPLETE")
print("=" * 60)

print(
    f"Source records       : {len(df_raw):,}"
)

print(
    f"Prepared documents   : {len(rag_documents):,}"
)

print(
    f"Unique document IDs  : "
    f"{len(set(document_ids)):,}"
)

print(
    f"Baseline text field  : {TEXT_COLUMN}"
)

print(
    "Alternative text     : NOT included"
)

print(
    "Bad text             : NOT included"
)

print(
    "Original dataset     : UNCHANGED"
)

print("=" * 60)
print("[SUCCESS] Document layer is ready.")
print("=" * 60)


# ------------------------------------------------------------
# 10. Memory cleanup
# ------------------------------------------------------------

del df_profile
gc.collect()

print("[INFO] Temporary profiling objects released.")

RAG DOCUMENT CREATION
[SUCCESS] Source dataset validation passed.
[SUCCESS] Created 100 RAG documents.
[SUCCESS] Document validation passed.
[INFO] Valid documents: 100

------------------------------------------------------------
PREPARED DOCUMENT STATISTICS
------------------------------------------------------------
Documents           : 100
Total characters    : 258,134
Average characters  : 2,581.34
Minimum characters  : 1,606
Maximum characters  : 3,730

------------------------------------------------------------
DOCUMENT PREVIEW
------------------------------------------------------------

Document ID : doc_001
Topic       : Setting Up a Mobile Device for Company Email
Text preview:
**Setting Up a Mobile Device for Company Email**

**Prerequisites:**

* Mobile device with a supported operating system (iOS, Android, or Windows)
* Company email account credentials
* Mobile device management (MDM) profile installed (if required by company policy)

**Step 1: Ensure Mobile Device Ma